In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/sales_data/sales_data.csv",
    header=True,
    inferSchema=True
)

display(df)

date,Sales
2020-01-01,104.96714153011233
2020-02-01,126.47449984543101
2020-03-01,155.49244128451457
2020-04-01,173.8017271355088
2020-05-01,152.38830787056
2020-06-01,136.94434471622247
2020-07-01,132.93498529793106
2020-08-01,102.6743472915291
2020-09-01,74.86112880857142
2020-10-01,81.13988615014537


In [0]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- Sales: double (nullable = true)



In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_bronze")
    

In [0]:
bronze_df = spark.table("sales_bronze")

display(bronze_df)


date,Sales
2020-01-01,104.96714153011233
2020-02-01,126.47449984543101
2020-03-01,155.49244128451457
2020-04-01,173.8017271355088
2020-05-01,152.38830787056
2020-06-01,136.94434471622247
2020-07-01,132.93498529793106
2020-08-01,102.6743472915291
2020-09-01,74.86112880857142
2020-10-01,81.13988615014537


In [0]:
from pyspark.sql.functions import col

silver_df = (
    spark.table("sales_bronze")
    .filter(col("date").isNotNull())
    .filter(col("Sales").isNotNull())
    .dropDuplicates()
)

display(silver_df)

date,Sales
2020-01-01,104.96714153011233
2020-02-01,126.47449984543101
2020-03-01,155.49244128451457
2020-04-01,173.8017271355088
2020-05-01,152.38830787056
2020-06-01,136.94434471622247
2020-07-01,132.93498529793106
2020-08-01,102.6743472915291
2020-09-01,74.86112880857142
2020-10-01,81.13988615014537


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_silver")

In [0]:
display(spark.table("sales_silver"))

date,Sales
2022-02-01,197.53779732567014
2022-08-01,182.094210416518
2022-09-01,147.9923289919702
2020-12-01,101.77127389286885
2021-09-01,128.49807464285075
2021-08-01,115.16267727236142
2022-05-01,217.2948832900339
2021-03-01,166.0520918640916
2021-12-01,126.4668038521511
2022-10-01,133.7086049961553


In [0]:
from pyspark.sql.functions import sum, avg, max, min

gold_df = (
    spark.table("sales_silver")
    .groupBy("date")
    .agg(
        sum("Sales").alias("total_sales"),
        avg("Sales").alias("average_sales"),
        max("Sales").alias("maximum_sales"),
        min("Sales").alias("minimum_sales")
    )
    .orderBy("date")
)

display(gold_df)

date,total_sales,average_sales,maximum_sales,minimum_sales
2020-01-01,104.96714153011233,104.96714153011233,104.96714153011233,104.96714153011233
2020-02-01,126.47449984543101,126.47449984543101,126.47449984543101,126.47449984543101
2020-03-01,155.49244128451457,155.49244128451457,155.49244128451457,155.49244128451457
2020-04-01,173.8017271355088,173.8017271355088,173.8017271355088,173.8017271355088
2020-05-01,152.38830787056,152.38830787056,152.38830787056,152.38830787056
2020-06-01,136.94434471622247,136.94434471622247,136.94434471622247,136.94434471622247
2020-07-01,132.93498529793106,132.93498529793106,132.93498529793106,132.93498529793106
2020-08-01,102.6743472915291,102.6743472915291,102.6743472915291,102.6743472915291
2020-09-01,74.86112880857142,74.86112880857142,74.86112880857142,74.86112880857142
2020-10-01,81.13988615014537,81.13988615014537,81.13988615014537,81.13988615014537


In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_gold")

In [0]:
display(spark.table("sales_gold"))

date,total_sales,average_sales,maximum_sales,minimum_sales
2020-01-01,104.96714153011233,104.96714153011233,104.96714153011233,104.96714153011233
2020-02-01,126.47449984543101,126.47449984543101,126.47449984543101,126.47449984543101
2020-03-01,155.49244128451457,155.49244128451457,155.49244128451457,155.49244128451457
2020-04-01,173.8017271355088,173.8017271355088,173.8017271355088,173.8017271355088
2020-05-01,152.38830787056,152.38830787056,152.38830787056,152.38830787056
2020-06-01,136.94434471622247,136.94434471622247,136.94434471622247,136.94434471622247
2020-07-01,132.93498529793106,132.93498529793106,132.93498529793106,132.93498529793106
2020-08-01,102.6743472915291,102.6743472915291,102.6743472915291,102.6743472915291
2020-09-01,74.86112880857142,74.86112880857142,74.86112880857142,74.86112880857142
2020-10-01,81.13988615014537,81.13988615014537,81.13988615014537,81.13988615014537


In [0]:
display(
    spark.table("sales_gold")
    .orderBy("date")
)

date,total_sales,average_sales,maximum_sales,minimum_sales
2020-01-01,104.96714153011233,104.96714153011233,104.96714153011233,104.96714153011233
2020-02-01,126.47449984543101,126.47449984543101,126.47449984543101,126.47449984543101
2020-03-01,155.49244128451457,155.49244128451457,155.49244128451457,155.49244128451457
2020-04-01,173.8017271355088,173.8017271355088,173.8017271355088,173.8017271355088
2020-05-01,152.38830787056,152.38830787056,152.38830787056,152.38830787056
2020-06-01,136.94434471622247,136.94434471622247,136.94434471622247,136.94434471622247
2020-07-01,132.93498529793106,132.93498529793106,132.93498529793106,132.93498529793106
2020-08-01,102.6743472915291,102.6743472915291,102.6743472915291,102.6743472915291
2020-09-01,74.86112880857142,74.86112880857142,74.86112880857142,74.86112880857142
2020-10-01,81.13988615014537,81.13988615014537,81.13988615014537,81.13988615014537
